In [3]:
import subprocess
from pathlib import Path
import json
import pandas as pd

In [ ]:
def xml2csv_multi_call(path: str | Path) -> None:
    xml2csv = Path(
        r"C:\Program Files (x86)\SUMO\tools\xml\xml2csv.py"
    )

    for xml_file in Path(path).glob("*.xml"):
        if xml_file.name.endswith("battery.xml"):
            continue
        subprocess.run(
            ["python", str(xml2csv), str(xml_file)],
            check=True,
        )
        print(f"Converted {xml_file}.")


xml2csv_multi_call(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\noDepot_run_2026-09-09-12-41-40\68_directory")
xml2csv_multi_call(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\noDepot_run_2026-09-09-12-41-40\67_directory")

## Not Working

In [ ]:

from datetime import date
from pathlib import Path

from analysis.output_files import SeedOutputFiles

from energy_storage_system.charging_station import ChargingStation
from energy_storage_system.energy_storage_system import EnergyStorageSystem

PROJECT_ROOT = Path(__file__).resolve().parent
SCENARIO_ROOT = PROJECT_ROOT.parent

SUMO_DIR = SCENARIO_ROOT / "sumo"
SUMO_OUTPUT_DIR = SUMO_DIR / "output"
EBUS_DIR = SCENARIO_ROOT / "eBuS"
FILES_DIR = EBUS_DIR / "files"
PV_DATA_DIR = EBUS_DIR / "pv_estimation/data"

@staticmethod
def run_energy_storage_system( run_dir: Path, start_date: date):
    """
    Build the energy storage system profile from a seed run's SUMO
    chargingstations output (a "<seed>_directory" folder, see
    tools.order_output.order_output) and write the result back as XML.
    Uses the PV data fetched for start_date by run_pvgis_api_call.
    """
    files = SeedOutputFiles(run_dir)
    chargingstations_file = files.get_file("chargingstations")
    output_file = files.output_dir / chargingstations_file.name.replace(
        "_chargingstations.xml", "_ess.xml"
    )
    pv_csv_path = PV_DATA_DIR / f"{start_date}_solar_power_v6_scaled.csv"

    EnergyStorageSystem(
        charging_stations=ChargingStation.from_xml(chargingstations_file),
        ess_factor=4.0,  # each station's ESS = ess_factor * that station's Peak Power (kWh)
        pv_csv_path=pv_csv_path,
        output_path=output_file,
        start_soc=0.2,  # fraction (0.0-1.0) of each station's own ESS capacity
        pv_factor=1.0,  # scales the PV power generated by each station
        grid_charge_max_soc=0.2,  # always draw from grid below this fraction of SoC
        grid_charge_power=50.0,  # power (kW) drawn from grid while below grid_charge_max_soc
        efficiency=0.95,  # battery round-trip efficiency (0.0-1.0)
    ).main()
    print(f"ESS output written to {output_file}")

if __name__ == "__main__":
    run_energy_storage_system(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\storage_run_2026-09-03-14-05-04\67_directory", "2024-08-22")
    run_energy_storage_system(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\storage_run_2026-09-03-14-05-04\68_directory", "2024-08-22")
    run_energy_storage_system(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\storage_run_2026-09-03-14-05-04\70_directory", "2024-08-22")
    run_energy_storage_system(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\storage_run_2026-09-03-14-05-04\75_directory", "2024-08-22")